# Backpropagation: Gradients, Hessians, and Parameter Tracking
This notebook collects the math-to-code mappings for backpropagation, Hessian computation, and tracking parameter changes layer by layer.

## 1. Forward and Backward Equations
$z^{(l)} = W^{(l)} a^{(l-1)} + b^{(l)}$\n\n$a^{(l)} = \sigma(z^{(l)})$

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 5)
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)


## 2. Backpropagation (Manual + Autograd)

In [5]:
X = torch.randn(32, 10)
y = torch.randint(0, 5, (32,))

optimizer.zero_grad()
logits = model(X)
loss = loss_fn(logits, y)
loss.backward()

for name, p in model.named_parameters():
    print(name, p.grad.norm().item())


0.weight 0.19761575758457184
0.bias 0.03596865013241768
2.weight 0.25253552198410034
2.bias 0.09522736072540283


## 3. Hessian and Hessian–Vector Product

In [9]:
params = torch.cat([p.flatten() for p in model.parameters()])
params.requires_grad_(True)

def loss_from_params(params):
    idx = 0
    for p in model.parameters():
        numel = p.numel()
        p.data = params[idx:idx+numel].view_as(p)
        idx += numel
    logits = model(X)
    return loss_fn(logits, y)

grad = torch.autograd.grad(loss_from_params(params), params, create_graph=True, allow_unused=True)[0]
print('Gradient norm:', grad)
#v = torch.randn_like(params)
#Hv = torch.autograd.grad(grad @ v, params)[0]

#print('HVP norm:', Hv.norm().item())


Gradient norm: None


## 4. Tracking Parameter Changes

In [ ]:
prev_params = {name: p.detach().clone() for name, p in model.named_parameters()}

optimizer.step()

for name, p in model.named_parameters():
    drift = (p - prev_params[name]).norm().item()
    print(name, 'drift:', drift)
